# Fixed-budget oracle

Let $v_i\ge0$ be the importance of row $i$, and let $\mathcal C(M)$ contain the maximal selected runs of a binary mask $M$. Define

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad
L(M)=\sum_{C\in\mathcal C(M)}T(|C|).$$

For a fixed budget $1\le R\le N$, the objective is

$$\boxed{\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.}$$

The oracle produces `oracle_reference.csv` for [selector comparison](02_03_selector_sweep.ipynb). Latencies are additive model estimates.

<a id="setup"></a>
## 1. Setup

Run from this directory. Set the row budgets below.

In [13]:
from dataclasses   import dataclass
from pathlib       import Path
from scipy.ndimage import maximum_filter1d
from tqdm.auto     import tqdm

import json

import numpy  as np
import pandas as pd

<a id="cost-model"></a>
## 2. Run latency follows a two-line model

For integer run length $\ell$,

$$T(0)=0,\qquad T(\ell)=b\ell+\delta\max(s,\ell)\quad(\ell\ge1).$$

Here $b$ is `short_slope`, $\delta$ is `slope_change`, and $s$ is `boundary`, all in row units. The short-run intercept is $\delta s$, and the slope beyond the boundary is $b+\delta$.

Assume $s\ge0$, $b\ge0$, $\delta\ge0$, and $b+\delta>0$. The boundary $s$ remains real-valued. Beyond the measured range, the model extends linearly.

In [14]:
@dataclass(frozen=True)
class TwoLineCost:

    """Two-line latency model in row units."""

    boundary     : float
    short_slope  : float
    slope_change : float

    def table(self, n):
        
        """Return the latency for each run length from 0 through n."""

        lengths = np.arange(n + 1)
        latency = (
              self.short_slope  * lengths
            + self.slope_change * np.maximum(self.boundary, lengths)
        )
        latency[0] = 0
        return latency

<a id="solver"></a>
## 3. The ratio objective reduces to additive subproblems

For a proposed ratio $q$, define

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

`bellman` computes $F_R(q)$; `recover` returns a maximizing mask. `solve` updates $q$ to the recovered ratio until the upper and lower bounds meet the tolerance. The [derivation](#derivations) gives the bounds and recurrence.

### Array helpers

`chunks` returns selected intervals $[\mathrm{start},\mathrm{end})$. `trailing_max` returns the maximum of the last `width` values at each position.

In [15]:
def chunks(mask):
    """Return selected chunks as [start, end) intervals."""

    padded = np.concatenate(([0], mask, [0]))
    change = np.diff(padded)

    starts = np.flatnonzero(change ==  1) # 0 -> 1
    ends   = np.flatnonzero(change == -1) # 1 -> 0

    return np.column_stack((starts, ends))

In [16]:
def trailing_max(values, width):

    """Return the maximum of the last width values at each position.

    Example: [-1, -2, -3, -4, -5] with width=3 gives [-1, -1, -1, -2, -3].
    """
    if width < 1:
        raise ValueError("width must be positive")

    return maximum_filter1d(
        values,
        size   = width,

        # maximize with indices [j - width + 1, j]
        origin = (width - 1) // 2,

        # assign -inf to invaild indices
        mode   = "constant",
        cval   = -np.inf,
    )


### One problem per trace

`SelectionProblem` stores the importance vector, cost model, prefix sums, and latency table once. `solve(R)` returns the selected mask and ratio bounds.

In [17]:
class SelectionProblem:

    """One trace and its cost model, prepared once for all row budgets.

    Create a new problem when the importance values or cost model change.
    R, q, DP, and masks belong to individual calculations.
    """

    def __init__(self, v, cost: TwoLineCost):

        # Importance
        self.v = np.array(v, dtype=float, copy=True)
        self.P = np.concatenate(([0.0], np.cumsum(self.v)))

        # Latency
        self.cost = cost
        self.T    = cost.table(len(self.v))

    def solve(self, R, rtol=1e-6):

        """Return the best mask and ratio bounds."""

        T = self.T

        # Ratio bounds
        mask  = self.best_run(R)
        lower = self.total_importance(mask) / T[R]
        upper = self.total_importance(self.top_r(R)) / T[R]

        # Fixed tolerance
        tolerance = rtol * lower

        while upper - lower > tolerance:

        # 1. MAXIMIZATION: I - qL
        
            q  = lower
            DP = self.bellman(R, q)

        # 2. LOWER BOUND

            # recover best mask
            candidate       = self.recover(R, q, DP)
            importance      = self.total_importance(candidate)
            latency         = self.total_latency(candidate)
            candidate_ratio = importance / latency

            # update
            if candidate_ratio > lower:
                mask  = candidate
                lower = candidate_ratio

        # 3. UPPER BOUND

            # residual: best value of I - qL
            residual = max(0.0, DP[-1, -1])
            upper    = min(upper, q + residual / T[R])

            # roundoff
            upper = max(upper, lower)

        # 4. STAGNATION

            if lower <= q and upper - lower > tolerance:
                raise RuntimeError("Dinkelbach iteration stalled.")

        # GOTO WHILE
            
        return mask, lower, upper

    def bellman(self, R, q):

        """Compute the best importance - q * latency for each DP state."""

        # Data
        P    = self.P
        T    = self.T
        cost = self.cost

        # Latency model
        short_slope  = cost.short_slope
        slope_change = cost.slope_change
        intercept    = slope_change * cost.boundary

        # DP boundary
        boundary = min(int(np.floor(cost.boundary)), R)

        # Row counts
        Z = len(self.v) - R  # Unselected rows

        t = np.arange(R + 1)  # Previous selected count
        r = np.arange(R + 1)  # Current selected count

        # DP[z, r]: best I - qL over the first z + r rows
        # z excluded, r selected
        DP    = np.empty((Z + 1, R + 1))
        DP[0] = P[:R + 1] - q * T[:R + 1]

        for z in range(1, Z + 1):

            # Last zero at z + t - 1; final chunk length r - t.
            A = DP[z - 1, t] - P[z + t] + q * short_slope * t
            B = DP[z - 1, t] - P[z + t] + q * (short_slope + slope_change) * t

            # Short chunk: length <= boundary
            short_max = np.full(R + 1, -np.inf)
            if boundary:
                short_max[1:] = trailing_max(A[:-1], boundary)

            # Long chunk: length > boundary
            long_max = np.full(R + 1, -np.inf)
            long_max[boundary + 1:] = np.maximum.accumulate(B)[:R - boundary]

            # Choose the best empty, short, or long final chunk.
            DP[z] = np.maximum.reduce([
                DP[z - 1],
                P[z + r] - q * (intercept + short_slope * r) + short_max,
                P[z + r] - q * (short_slope + slope_change) * r + long_max,
            ])

        return DP

    def recover(self, R, q, DP):
        """Recover a mask maximizing I - qL."""

        P = self.P
        T = self.T
        Z = len(self.v) - R

        mask = np.zeros(len(self.v), dtype=bool)
        r = R

        for z in range(Z, 0, -1):

            # Last zero
            t = np.arange(r + 1)
            scores = (
                DP[z - 1, t]           # Prefix score
                + P[z + r] - P[z + t]  # Chunk importance
                - q * T[r - t]         # Latency penalty
            )
            best_t = int(np.argmax(scores))

            # Final chunk
            mask[z + best_t:z + r] = True

            # Previous state
            r = best_t

        # Remaining prefix: all selected
        mask[:r] = True
        return mask

    def best_run(self, R):
        """Select R consecutive rows with maximum importance."""

        P = self.P

        # Chunk importance
        importance = P[R:] - P[:-R]

        # Best chunk
        start                 = int(np.argmax(importance))
        mask                  = np.zeros(len(self.v), dtype=bool)
        mask[start:start + R] = True

        return mask

    def top_r(self, R):
        """Select the R rows with greatest importance."""

        # Importance ranking
        order = np.argsort(-self.v, kind="stable")

        # Top R rows
        mask            = np.zeros(len(self.v), dtype=bool)
        mask[order[:R]] = True

        return mask

    def total_importance(self, mask):

        """Return the total importance of selected rows."""
        return self.v[mask].sum()

    def total_latency(self, mask):

        """Return the total latency of selected runs."""
        spans   = chunks(mask)
        lengths = spans[:, 1] - spans[:, 0]
        return self.T[lengths].sum()

<a id="inputs"></a>
## 4. Experiment inputs

Read all activation traces from [02_01](02_01_capture_traces.ipynb) in their saved order and the fitted latency model from experiment 01.

Each trace stores its dimensions as numeric `n` (input channels) and `d` (output channels).

### Traces and model

In [18]:
MODEL = Path("../01/latency_model.json")
TRACE = Path(  "activation_traces.json")

In [19]:
traces = json.loads(TRACE.read_text())["traces"]
for trace in traces:
    trace["values"] = np.asarray(trace["values"], dtype=float)

In [ ]:
model_data = json.loads(MODEL.read_text())
model      = model_data["models"]["Two-line (free)"]

# The experiment uses the two-line fit with a zero tail intercept.
boundary_kib     = model["boundary_kib"]
short_slope_kib  = model["short_slope_ms_per_kib"]
slope_change_kib = model["slope_change_ms_per_kib"]

<a id="row-units"></a>
### Row-unit conversion

Use the captured importance values directly. The solver stores and accumulates them as float64.

The capture stores the byte size of one weight row as `row_bytes`. In KiB, $w=\mathrm{row\_bytes}/1024$. For the fitted model

$$T_{\mathrm{KiB}}(z)=b_{\mathrm{KiB}}z+\delta_{\mathrm{KiB}}\max(z_{\mathrm{2L}},z),$$

substituting $z=w\ell$ gives

$$s=\frac{z_{\mathrm{2L}}}{w},\qquad b=wb_{\mathrm{KiB}},\qquad \delta=w\delta_{\mathrm{KiB}}.$$

Here $z_{\mathrm{2L}}$ is `boundary_kib` in `models["Two-line (free)"]`. The JSON stores $b_{\mathrm{KiB}}$ as `short_slope_ms_per_kib` and $\delta_{\mathrm{KiB}}$ as `slope_change_ms_per_kib`. Both coefficients are multiplied by the row length to obtain ms per row.

In [21]:
cases = []

for trace in traces:
    
    # Importance
    v = trace["values"]
    
    # KiB per row
    w = trace["row_bytes"] / 1024

    # Cost model: KiB -> rows
    cost = TwoLineCost(
        boundary     = boundary_kib / w, 
        short_slope  = w * short_slope_kib, 
        slope_change = w * slope_change_kib
    )

    # Selection problem
    problem = SelectionProblem(v, cost)
    cases.append((trace, problem))

<a id="run"></a>
## 5. Oracle reference

For each trace and budget fraction $f$, select $R=\max(1,\lfloor fN\rfloor)$ rows.
Each dictionary below is one CSV row. Its keys are the column names.
Save the measurements, bounds, and inputs needed by the report; [02_04](02_04_analyze_oracle.ipynb) computes percentages, gaps, and fragmentation.

`importance_total` is the sum of the captured input importance. `short_slope_ms_per_row` and `slope_change_ms_per_row` store the initial slope and its change at the boundary. The slope change is the cost of one row of short-run deficit. `intervals` stores selected `[start, end)` pairs as JSON text in one CSV cell.

In [22]:
# Requested fractions of rows, from 0.1 through 0.9.
BUDGETS = np.arange(1, 10) / 10

records = []
for trace, problem in tqdm(cases, desc="Oracle reference", unit="trace"):
    for fraction in BUDGETS:

        # decide R using fraction
        R = max(1, int(fraction * len(problem.v)))

        # optimal mask & its (importance)/(latency) ratio
        mask, lower, upper = problem.solve(R)

        records.append({
            # rewrite trace infomation
            "trace_id" : trace["trace_id"],
            "n"        : trace["n"],
            "d"        : trace["d"],

            # fraction
            "fraction"  : fraction,
            "R_nominal" : R,

            # total importance
            "importance_total" : problem.v.sum(),

            # latency model
            "saturation_rows"          : problem.cost.boundary,
            "short_slope_ms_per_row"   : problem.cost.short_slope,
            "slope_change_ms_per_row"  : problem.cost.slope_change,

            # optimal mask
            "importance" : problem.total_importance(mask),
            "latency"    : problem.total_latency   (mask),
            "eta_lower"  : lower,
            "eta_upper"  : upper,
            "intervals"  : json.dumps(chunks(mask).tolist()),
        })

Oracle reference: 100%|██████████| 504/504 [03:27<00:00,  2.43trace/s]


### Check the reference

In [23]:
reference = pd.DataFrame(records)
assert not reference.duplicated(["trace_id", "fraction"]).any()
reference.head()

,trace_id,n,d,fraction,R_nominal,importance_total,saturation_rows,short_slope_ms_per_row,slope_change_ms_per_row,importance,latency,eta_lower,eta_upper,intervals
0,0,896,896,0.1,89,52.812195,118.857143,0.000214,0.000103,7.156982,0.031269,228.880808,228.880808,"[[483, 572]]"
1,0,896,896,0.2,179,52.812195,118.857143,0.000214,0.000103,14.249634,0.062753,227.075004,227.075004,"[[13, 86], [481, 587]]"
2,0,896,896,0.3,268,52.812195,118.857143,0.000214,0.000103,20.660400,0.094022,219.738998,219.738998,"[[35, 86], [483, 585], [670, 785]]"
3,0,896,896,0.4,358,52.812195,118.857143,0.000214,0.000103,25.059204,0.113432,220.917361,220.917361,"[[8, 127], [483, 603], [670, 789]]"
4,0,896,896,0.5,448,52.812195,118.857143,0.000214,0.000103,30.990967,0.144798,214.028253,214.028253,"[[13, 108], [313, 432], [483, 602], [670, 785]]"


### Save the reference

Replace `oracle_reference.csv`. After changing the traces, cost model, or oracle settings, rerun this notebook, 02_03, and 02_04 in order. Files are not linked by hashes.

In [24]:
RESULTS = Path("oracle_reference.csv")
reference.to_csv(RESULTS, index=False)

<a id="derivations"></a>
## 6. Derivation

<a id="bounds"></a>
### A. Feasible ratios and residuals bound the optimum

Every mask selecting $R$ rows satisfies

$$L(M)=bR+\delta\sum_C\max(s,|C|)\ge T(R).$$

Let $M_C$ be the most important contiguous run of length $R$, and $M_T$ select the $R$ largest values. Then

$$\ell_0=\frac{I(M_C)}{T(R)}\le\eta_R^\star\le u_0=\frac{I(M_T)}{T(R)}.$$

At $q=\ell$, recover a maximizer $M_q$ of $I-qL$. Since $F_R(q)\ge0$,

$$\boxed{\ell\leftarrow\frac{I(M_q)}{L(M_q)}=q+\frac{F_R(q)}{L(M_q)},\qquad
u\leftarrow\min\left(u,q+\frac{F_R(q)}{T(R)}\right).}$$

The upper bound follows from $I(M^\star)-qL(M^\star)\le F_R(q)$ and $L(M^\star)\ge T(R)$.

In exact arithmetic, $F_R(q)>0$ strictly improves the feasible ratio; $F_R(q)=0$ makes both bounds equal to the optimum. Finitely many masks imply finite termination. For nonzero importance, stopping at $u-\ell\le\epsilon\ell_0$ guarantees

$$0\le1-\frac{\ell}{\eta_R^\star}\le\epsilon.$$

Zero importance returns immediately. The implementation uses float64 with `rtol = 1e-6`; the guarantees above assume exact arithmetic.

<a id="recurrence"></a>
### B. The last excluded row defines the recurrence

Let $Z=N-R$ and $P_j=\sum_{i<j}v_i$. Define $D_{z,r}$ as the best $I-qL$ over the first $z+r$ rows, with $z$ excluded and $r$ selected. The base case is

$$D_{0,r}=P_r-qT(r).$$

Place the last excluded row after a prefix with $z-1$ excluded and $t$ selected rows. The final run is $[z+t,z+r)$, possibly empty. Therefore,

$$\boxed{D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

Every feasible prefix has this decomposition, so $F_R(q)=D_{Z,R}$.

<a id="range-maxima"></a>
### C. Two range maxima evaluate each DP row in linear time

For fixed $z$, define

$$A_{z,t}=D_{z-1,t}-P_{z+t}+qbt,\qquad
B_{z,t}=D_{z-1,t}-P_{z+t}+q(b+\delta)t.$$

Let $h=\min(\lfloor s\rfloor,R)$ (`boundary` in code); the intercept $a=\delta s$ is not rounded. Separating empty, short, and long final runs gives

$$D_{z,r}=\max\left\{\begin{aligned}
&D_{z-1,r},\\
&P_{z+r}-q(a+br)+\max_{\max(0,r-h)\le t<r}A_{z,t},\\
&P_{z+r}-q(b+\delta)r+\max_{0\le t<r-h}B_{z,t}.
\end{aligned}\right.$$

Empty maxima equal $-\infty$. `trailing_max` computes the short-run window; `np.maximum.accumulate` computes the long-run prefix. Together they evaluate a row in $O(R+1)$ time, replacing a separate scan over $t$ at every state.

<a id="recovery"></a>
### D. Recovery preserves the time bound

From $(Z,R)$, choose a maximizing $t$, select $[z+t,z+r)$, and continue at $(z-1,t)$. At $z=0$, select the remaining prefix.

Bellman evaluation and recovery together require

$$\boxed{O((N-R+1)(R+1))\text{ time and memory per additive solve}.}$$

For $K$ additive solves, total time after initialization is $O(K(N-R+1)(R+1))$.

<a id="reporting-identities"></a>
### E. Short runs account for the latency overhead

Define the short-run deficit, with $(x)_+=\max(x,0)$, as

$$D(M)=\sum_{C\in\mathcal C(M)}(s-|C|)_+.$$

Summing $T(|C|)=(b+\delta)|C|+\delta(s-|C|)_+$ gives

$$\boxed{L(M)=(b+\delta)\|M\|_1+\delta D(M).}$$

`shortfall_rows` is $D(M)$; `overhead_ms` is $\delta D(M)$. `short_chunks` counts runs shorter than $s$.

#### Ratio gaps at the same budget

For a mask selecting exactly $R$ rows, let $u_M=I(M)/L(M)$. If $\ell>0$, the oracle bounds imply

$$\max(0,100(1-u_M/\ell))
\le g(M)=100\left(1-\frac{u_M}{\eta_R^\star}\right)
\le\max(0,100(1-u_M/u)).$$

`gap_upper_pct` reports the upper end. Gaps are undefined when the selected count differs from $R$ or the trace has zero importance.